# 📊 Notebook 01 — Data Preprocessing
## Bitcoin Sentiment × Trader Performance Analysis

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded')

## 1 & 2 · Load & Merge Datasets

In [ ]:
trader = pd.read_csv('../data/historical_data.csv')
fg = pd.read_csv('../data/fear_greed_index.csv')

# Parse dates
trader['date'] = pd.to_datetime(trader['Timestamp IST'], format='%d-%m-%Y %H:%M').dt.strftime('%Y-%m-%d')
fg['date'] = pd.to_datetime(fg['date']).dt.strftime('%Y-%m-%d')

# Filter to 2024
trader = trader[trader['date'].str.startswith('2024')]
fg_2024 = fg[fg['date'].str.startswith('2024')]

# Merge
merged = pd.merge(trader, fg_2024[['date', 'classification', 'value']], on='date', how='left')
merged.rename(columns={'value': 'sentiment_value', 'classification': 'sentiment_class'}, inplace=True)
print(f'Merged shape: {merged.shape}')

## 3 · Filter Closed Trades

In [ ]:
closed = merged[merged['Closed PnL'] != 0].copy()
print(f'Closed trades: {len(closed):,}')

## 4 · Feature Engineering

In [ ]:
# Base features
closed['Net PnL'] = closed['Closed PnL'] - closed['Fee']
closed['Is Profitable'] = (closed['Closed PnL'] > 0).astype(int)

# Trade Type mapping
closed['Trade Type'] = closed['Direction'].map({
    'Open Long': 'Long', 'Close Long': 'Long',
    'Open Short': 'Short', 'Close Short': 'Short',
    'Buy': 'Spot', 'Sell': 'Spot'
}).fillna('Other')

closed['PnL per USD'] = closed['Closed PnL'] / (closed['Size USD'] + 1e-6)
closed['Hour'] = pd.to_datetime(closed['Timestamp IST'], format='%d-%m-%Y %H:%M').dt.hour
closed['Is Extreme'] = closed['sentiment_class'].str.contains('Extreme', na=False)

# Sentiment Score Bucket
closed['Sentiment Score Bucket'] = pd.cut(
    closed['sentiment_value'],
    bins=[-1, 24, 49, 50, 74, 100],
    labels=['0-24', '25-49', '50', '51-74', '75-100']
)

# Lag sentiment
fg_daily = fg_2024.drop_duplicates('date').set_index('date').sort_index()
fg_daily['lag1'] = fg_daily['value'].shift(1)
fg_daily['lag3'] = fg_daily['value'].shift(3)
fg_daily['lag7'] = fg_daily['value'].shift(7)
closed = pd.merge(closed, fg_daily[['lag1', 'lag3', 'lag7']], left_on='date', right_index=True, how='left')

# Outliers
q01 = closed['Closed PnL'].quantile(0.01)
q99 = closed['Closed PnL'].quantile(0.99)
closed_clean = closed[(closed['Closed PnL'] >= q01) & (closed['Closed PnL'] <= q99)].copy()

print("Features added successfully.")


## 5 · Export

In [ ]:
merged.to_csv('../data/merged_all_trades.csv', index=False)
closed.to_csv('../data/closed_trades.csv', index=False)
closed_clean.to_csv('../data/closed_trades_clean.csv', index=False)
print('Data exported.')